# Retrieving Complete Bacterial Taxonomic Lineages from NCBI Using Python (Biopython Entrez)

This Python notebook intends to demonstrate pragrammatric approach to retrive full taxonomy lineage from NCBI, with a use case of a bunch of bacteria genus commonly resident in fermented foods.

The scritps is re-usable and adaptable for various cases in microbiology research. 

## Steps

1. Define a function to retrive full taxonomy lineage based on known species rank
2. Example input data (fermenting bacteria)
3. Obtain full lineage output using loop iteration


## Requirements

- Python 3.9+
- `biopython` and `pandas`

```
pip install biopython pandas
```

In [38]:
from Bio import Entrez
import pandas as pd
import os

In [39]:
def get_taxonomy_lineage(taxon_name: str, rank: str = "Genus") -> list | None:
    """Fetches complete taxonomy lineage based on the known taxonomic level (the highest resolution available) from the NCBI taxonomy database.

    Args:
        rank (str): The taxonomic rank to search for (default is "Genus"). "Family", "Order", "Class", "Phylum", or "Kingdom" can also be used.
        taxon_name (str): The name of the taxon to search for. If None, the function will not perform a search.

    Returns:
        list: A list containing the lineage of the taxon, or None if not found.
        The lineage includes domain, kingdom (or clade), phylum, class, order, and family.
        If the taxon is not found, it returns None.
        If the rank is invalid, it returns None with an error message.
        If the taxon_name is None, it returns None with an error message.
    """
    
    #--- Input validation ---
    # Check if taxon_name is provided
    if taxon_name is None:
        print("Taxon name is None. Please provide a valid taxon name.")
        return None
    # Check if rank is valid
    if rank not in ["Genus", "Family", "Order", "Class", "Phylum", "Kingdom"]:
        print(f"Invalid rank: {rank}. Please use one of the following ranks: Genus, Family, Order, Class, Phylum, Kingdom.")
        return None


    #--- Set the email for NCBI Entrez ---
    # This is required by NCBI to track usage and for contact in case of issues
    # Replace with your email address
    Entrez.email = os.getenv("NCBI_EMAIL")


    #--- Search for the genus in the NCBI taxonomy database ---
    # The search term is formatted to include the genus name followed by "[Genus]" to specify the search field
    # This ensures that the search is limited to the genus level in the taxonomy database
    handle = Entrez.esearch(db="taxonomy", term=f"{taxon_name}[{rank}]")
    record = Entrez.read(handle) # Read the search results
    handle.close() # Close the handle to free resources


    #--- Check if any IDs were found for the genus ---
    # If no IDs are found, print a message and return None
    # This is important to handle cases where the genus does not exist in the database
    if not record["IdList"]:
        print(f"No taxonomy ID found for {rank}: {taxon_name}") 
        return None


    #--- Fetch the taxonomy record using the first ID found ---
    # The first ID in the IdList is used to fetch the complete taxonomy record
    # This is because the search may return multiple IDs, but we are interested in the first one
    # The efetch function retrieves the record in XML format for easier parsing
    # The record contains detailed information about the taxonomy, including lineage
    # The lineage includes domain, kingdom (or clade), phylum, class, order, and family
    taxid = record["IdList"][0] # Get the first taxonomy ID from the search results
    handle = Entrez.efetch(db="taxonomy", id=taxid, retmode="xml") 
    records = Entrez.read(handle) # Read the fetched record
    handle.close() # Close the handle to free resources

    #--- Extract the lineage from the fetched record ---
    # The lineage is extracted from the first record in the list of records returned by efetch
    # The lineage is a string that includes the complete taxonomy hierarchy for the genus
    # It is formatted as "domain; kingdom; phylum; class; order; family"
    lineage = records[0]["Lineage"] # Extract the lineage from the record, including domain, kingdom (or clade), phylum, class, order, and family
    if not lineage == None:
        lineage_list = lineage.split("; ") # Split the lineage into a list
    
    return lineage_list

This function is particularly useful for microbiology research where you need to understand the evolutionary relationships and taxonomic classification of bacterial genera commonly found in fermented food and gut microbiome.

In [42]:
# Test the function with a specific genus
genus = "Lactobacillus"
lineage = get_taxonomy_lineage(rank="Genus", taxon_name=genus)
print(f"Lineage for {genus}: {lineage}")

Lineage for Lactobacillus: ['cellular organisms', 'Bacteria', 'Bacillati', 'Bacillota', 'Bacilli', 'Lactobacillales', 'Lactobacillaceae']


### Example

In [3]:
# Define a list of target genera: Fermenting bacteria
genera = ['Acetobacter', 'Gluconacetobacter', 'Lentibacillus', 'Brevibacterium', 'Erwinia', 'Enterobacter', 'Pantoea', 
          'Kosakonia', 'Lactobacillus', 'Companilactobacillus', 'Schleiferilactobacillus', 'Ligilactobacillus', 
          'Lactiplantibacillus', 'Loigolactobacillus', 'Paucilactobacillus', 'Limosilactobacillus', 'Fructilactobacillus', 
          'Acetilactobacillus', 'Secundilactobacillus', 'Lentilactobacillus', 'Carnobacterium', 'Weissella', 'Oenococcus', 
          'Enterococcus', 'Tetragenococcus', 'Streptococcus', 'Lactococcus', 'Pediococcus', 'Periweissella', 'Leuconostoc', 
          'Marinilactobacillus', 'Alkalibacterium', 'Eggerthella', 'Propionibacterium', 'Staphylococcus', 'Kocuria']

len(genera)


36

In [43]:
# Initialize an empty DataFrame to store all lineages
df_all_lineages = pd.DataFrame()

# Loop through each genus and get the taxonomy lineage
for genus in genera:
    print(f"Processing genus: {genus}")
    lineage = get_taxonomy_lineage(rank="Genus", taxon_name=genus)
    
    # Skip if lineage is None or does not belong to Bacteria
    if not lineage or lineage[1] != "Bacteria":
        print(f"Skipping genus: {genus} (No lineage or not Bacteria)")
        continue

    # Create a DataFrame for the current genus
    try:
        df_genus_lineage = pd.DataFrame([lineage[1:]], columns=['Domain', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family'])
        df_genus_lineage['Genus'] = genus
        
        # Append to the main DataFrame
        df_all_lineages = pd.concat([df_all_lineages, df_genus_lineage], ignore_index=True)
    except Exception as e:
        print(f"Error processing genus {genus}: {e}")

# Print summary of successful retrievals
genus_successful = df_all_lineages["Genus"].tolist()
print(f"{len(genus_successful)} out of {len(genera)} genera were successfully retrieved.")

# Display the first few rows of the DataFrame
df_all_lineages.head()

Processing genus: Acetobacter


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/Entrez/__init__.py:734: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


Processing genus: Gluconacetobacter


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/Entrez/__init__.py:734: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


Processing genus: Lentibacillus


...

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/Entrez/__init__.py:734: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


Processing genus: Kocuria


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/Bio/Entrez/__init__.py:734: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


36 out of 36 genera were successfully retrieved.


,Domain,Kingdom,Phylum,Class,Order,Family,Genus
0,Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Acetobacterales,Acetobacteraceae,Acetobacter
1,Bacteria,Pseudomonadati,Pseudomonadota,Alphaproteobacteria,Acetobacterales,Acetobacteraceae,Gluconacetobacter
2,Bacteria,Bacillati,Bacillota,Bacilli,Bacillales,Bacillaceae,Lentibacillus
3,Bacteria,Bacillati,Actinomycetota,Actinomycetes,Micrococcales,Brevibacteriaceae,Brevibacterium
4,Bacteria,Pseudomonadati,Pseudomonadota,Gammaproteobacteria,Enterobacterales,Erwiniaceae,Erwinia
